# DS1 · Inferencia causal — Práctica guiada

**Caso TiendaSur.** Dirección te hace tres preguntas que parecen distintas y son la misma: *¿cuánto vende de más la app?*, *¿sirvió el nuevo escaparate de las tiendas piloto?*, *¿a quién enviamos el cupón?* Los datos están **simulados con el efecto causal conocido**: así puedes comprobar qué método lo recupera y cuál se equivoca (y por cuánto).

Todo corre en CPU en pocos minutos y no necesita clave de API.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.2f}".format

def coef(modelo, nombre):
    """Coeficiente, error estándar e IC 95 % de un modelo de statsmodels."""
    ic = modelo.conf_int().loc[nombre]
    return modelo.params[nombre], modelo.bse[nombre], ic[0], ic[1]

def informe(etiqueta, estimacion, verdad, ic=None):
    """Una línea comparable: estimación, verdad conocida y sesgo."""
    extra = f"  IC95% [{ic[0]:6.2f}, {ic[1]:6.2f}]" if ic is not None else ""
    print(f"{etiqueta:34s} estimación {estimacion:7.2f} | verdad {verdad:6.2f} | sesgo {estimacion - verdad:+7.2f}{extra}")

print("Entorno listo")

## DS1.1 · DAGs y el criterio de la puerta trasera

Un **DAG** (grafo acíclico dirigido) dibuja lo que crees sobre cómo se generan los datos: cada flecha es «causa directa». El **efecto causal de T sobre Y** se identifica ajustando por un conjunto Z tal que:
1. Z bloquea todos los caminos de *puerta trasera* (los que empiezan con una flecha que **entra** en T).
2. Z no contiene descendientes de T (no bloqueas el efecto que quieres medir ni abres caminos espurios).

La función de abajo implementa las reglas de bloqueo para que veas el criterio funcionar, no solo leerlo.

In [ ]:
def descendientes(aristas, nodo):
    """Todos los nodos alcanzables siguiendo las flechas desde `nodo`."""
    vistos, pila = set(), [nodo]
    while pila:
        actual = pila.pop()
        for a, b in aristas:
            if a == actual and b not in vistos:
                vistos.add(b)
                pila.append(b)
    return vistos

def caminos_puerta_trasera(aristas, tratamiento, resultado):
    """Caminos (sin importar el sentido de las flechas) de T a Y cuya primera flecha ENTRA en T."""
    vecinos = {}
    for a, b in aristas:
        vecinos.setdefault(a, set()).add(b)
        vecinos.setdefault(b, set()).add(a)
    encontrados = []
    def explorar(camino):
        ultimo = camino[-1]
        if ultimo == resultado:
            encontrados.append(camino)
            return
        for v in vecinos.get(ultimo, ()):
            if v not in camino:
                explorar(camino + [v])
    explorar([tratamiento])
    return [c for c in encontrados if (c[1], c[0]) in set(aristas)]   # primera flecha: c[1] -> T

def esta_bloqueado(aristas, camino, Z):
    """Reglas de d-separación: una cadena/bifurcación se bloquea condicionando en el nodo intermedio;
    un colisionador (a -> m <- b) está bloqueado salvo que condiciones en él o en un descendiente."""
    flechas = set(aristas)
    for i in range(1, len(camino) - 1):
        ant, m, sig = camino[i - 1], camino[i], camino[i + 1]
        es_colisionador = (ant, m) in flechas and (sig, m) in flechas
        if es_colisionador:
            if not ({m} | descendientes(aristas, m)) & set(Z):
                return True            # colisionador sin condicionar: bloquea
        elif m in Z:
            return True                # cadena o bifurcación condicionada: bloquea
    return False

def criterio_puerta_trasera(aristas, T, Y, Z=()):
    """¿Es Z un conjunto de ajuste válido para el efecto causal de T sobre Y?"""
    Z = set(Z)
    prohibidos = Z & descendientes(aristas, T)
    abiertos = [c for c in caminos_puerta_trasera(aristas, T, Y) if not esta_bloqueado(aristas, c, Z)]
    valido = not prohibidos and not abiertos
    print(f"Z = {sorted(Z) or '{}'}: {'VÁLIDO' if valido else 'NO válido'}"
          + (f" | descendientes de {T} en Z: {sorted(prohibidos)}" if prohibidos else "")
          + (f" | puertas traseras abiertas: {[' - '.join(c) for c in abiertos]}" if abiertos else ""))
    return valido

# DAG de TiendaSur: ¿cuánto gasto adicional produce usar la app?
ARISTAS = [
    ("fidelidad", "app"), ("fidelidad", "gasto"),      # confusor: el cliente fiel usa la app y gasta más
    ("app", "pedidos"), ("pedidos", "gasto"),          # mediador: la app aumenta pedidos y estos el gasto
    ("app", "gasto"),                                  # efecto directo
    ("app", "valoracion"), ("gasto", "valoracion"),    # colisionador: reseña influida por ambos
]
print("Caminos de puerta trasera app -> gasto:")
for c in caminos_puerta_trasera(ARISTAS, "app", "gasto"):
    print("  ", " - ".join(c))
for Z in [(), ("fidelidad",), ("fidelidad", "pedidos"), ("fidelidad", "valoracion")]:
    criterio_puerta_trasera(ARISTAS, "app", "gasto", Z)

## DS1.2 · Confusores, mediadores y colisionadores: tres papeles, tres decisiones opuestas

| Papel | Estructura | ¿Ajustar? |
|---|---|---|
| Confusor | Z → X, Z → Y | **Sí**: si no, sesgo |
| Mediador | X → M → Y | **Depende** de la pregunta: total (no) o directo (sí) |
| Colisionador | X → C ← Y | **No**: ajustar *crea* sesgo |

Simulamos los tres con un efecto total conocido de 2.

In [ ]:
rng = np.random.default_rng(1)
N = 20_000
EFECTO = 2.0

# --- Confusor: Z -> X, Z -> Y, X -> Y ------------------------------------------------------
z = rng.normal(size=N)
x = 0.8 * z + rng.normal(size=N)
y = EFECTO * x + 3.0 * z + rng.normal(size=N)
d1 = pd.DataFrame({"x": x, "y": y, "z": z})

# --- Mediador: X -> M -> Y y X -> Y directo (1) + indirecto (1 · 1) = total 2 ----------------
x = rng.normal(size=N)
m = 1.0 * x + rng.normal(size=N)
y = 1.0 * x + 1.0 * m + rng.normal(size=N)
d2 = pd.DataFrame({"x": x, "m": m, "y": y})

# --- Colisionador: X -> C <- Y, y X -> Y = 2 -------------------------------------------------
x = rng.normal(size=N)
y = EFECTO * x + rng.normal(size=N)
c = 1.0 * x + 1.0 * y + rng.normal(size=N)
d3 = pd.DataFrame({"x": x, "y": y, "c": c})

filas = [
    ("Confusor", "sin ajustar", smf.ols("y ~ x", d1).fit(), EFECTO),
    ("Confusor", "ajustando por Z (correcto)", smf.ols("y ~ x + z", d1).fit(), EFECTO),
    ("Mediador", "sin ajustar (efecto total)", smf.ols("y ~ x", d2).fit(), EFECTO),
    ("Mediador", "ajustando por M (solo directo)", smf.ols("y ~ x + m", d2).fit(), EFECTO),
    ("Colisionador", "sin ajustar (correcto)", smf.ols("y ~ x", d3).fit(), EFECTO),
    ("Colisionador", "ajustando por C (error)", smf.ols("y ~ x + c", d3).fit(), EFECTO),
]
tabla = pd.DataFrame([{"caso": a, "modelo": b, "estimación de X": coef(m_, "x")[0], "efecto total real": v}
                      for a, b, m_, v in filas])
tabla["sesgo"] = tabla["estimación de X"] - tabla["efecto total real"]
tabla

**Lee la tabla.** Sin ajustar, el confusor infla el efecto; ajustando por él se recupera 2. Con el mediador, ajustar «mata» la parte indirecta (queda 1, el efecto directo). Con el colisionador, lo *correcto* es no ajustar: ajustar por C convierte un efecto de +2 en uno de signo distinto. «Meter más variables» **no** es una estrategia segura.

## DS1.3 · Datos observacionales: el efecto de usar la app

Los clientes **deciden** si usan la app, y los más fieles y activos la usan más. Eso es el confusor. Efecto real simulado: **+8 €** de gasto por trimestre.

In [ ]:
def generar_clientes(n=20_000, semilla=7, fuerza_seleccion=1.0, efecto=8.0):
    """Clientes de TiendaSur. `app` = usa la app de pedidos (NO es aleatorio: se autoselecciona).
    `gasto` = gasto (€) en el trimestre siguiente. El efecto causal de la app es `efecto` para todos."""
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "antiguedad": rng.integers(0, 72, n),
        "gasto_previo": rng.lognormal(3.6, 0.5, n),
        "pedidos_previos": rng.poisson(5, n),
        "edad": rng.integers(18, 75, n),
        "urbano": rng.binomial(1, 0.6, n),
    })
    logit = fuerza_seleccion * (-2.6 + 0.02 * df["gasto_previo"] + 0.02 * df["antiguedad"]
                                + 0.12 * df["pedidos_previos"] + 0.6 * df["urbano"] - 0.025 * (df["edad"] - 40))
    df["propension_real"] = 1 / (1 + np.exp(-logit))
    df["app"] = rng.binomial(1, df["propension_real"])
    df["gasto"] = (30 + 0.8 * df["gasto_previo"] + 0.3 * df["antiguedad"] + 2.5 * df["pedidos_previos"]
                   + 5 * df["urbano"] + efecto * df["app"] + rng.normal(0, 18, n))
    return df

df = generar_clientes()
COVARIABLES = ["antiguedad", "gasto_previo", "pedidos_previos", "edad", "urbano"]
VERDAD = 8.0
print(f"Clientes: {len(df):,} | usan la app: {df['app'].mean():.1%}")
naive = df.loc[df.app == 1, "gasto"].mean() - df.loc[df.app == 0, "gasto"].mean()
informe("Diferencia de medias (ingenua)", naive, VERDAD)
df.groupby("app")[COVARIABLES].mean().T.rename(columns={0: "no usa app", 1: "usa app"})

## DS1.4 · Regresión con ajuste

Si conoces los confusores y la relación es aproximadamente lineal, incluirlos en la regresión basta.

In [ ]:
m_ols = smf.ols("gasto ~ app + antiguedad + gasto_previo + pedidos_previos + edad + urbano", df).fit(cov_type="HC1")
b, se, lo, hi = coef(m_ols, "app")
informe("Regresión con ajuste (OLS)", b, VERDAD, (lo, hi))

## DS1.5 · *Propensity score*: emparejamiento e IPW

El **propensity score** e(x) = P(T=1 | X=x) resume todas las covariables en un número. Dos formas de usarlo:
- **Emparejamiento:** cada cliente con app se compara con uno sin app *de propensión parecida*.
- **IPW** (ponderación por el inverso de la probabilidad): da más peso a quienes tenían pocas probabilidades de recibir el tratamiento que recibieron, para que ambos grupos se parezcan a la población.

Antes de estimar, dos comprobaciones: **solapamiento** y **equilibrio**.

In [ ]:
# 1) Propensity score: P(app = 1 | covariables)
escalador = StandardScaler().fit(df[COVARIABLES])
Xs = escalador.transform(df[COVARIABLES])
modelo_ps = LogisticRegression(max_iter=1000).fit(Xs, df["app"])
df["ps"] = modelo_ps.predict_proba(Xs)[:, 1]
print("Correlación ps estimado vs. real:", round(np.corrcoef(df["ps"], df["propension_real"])[0, 1], 3))

# 2) Solapamiento (overlap): ¿hay clientes parecidos en ambos grupos?
fig, ax = plt.subplots()
ax.hist(df.loc[df.app == 0, "ps"], bins=40, alpha=.6, label="no usa app", density=True)
ax.hist(df.loc[df.app == 1, "ps"], bins=40, alpha=.6, label="usa app", density=True)
ax.set(xlabel="propensity score", ylabel="densidad", title="Solapamiento"); ax.legend(); plt.show()

# 3) Equilibrio de covariables: diferencia media tipificada (SMD); |SMD| < 0,1 se considera equilibrado
def smd(x, t, w=None):
    w = np.ones(len(x)) if w is None else np.asarray(w)
    m1 = np.average(x[t == 1], weights=w[t == 1]); m0 = np.average(x[t == 0], weights=w[t == 0])
    return (m1 - m0) / np.sqrt((x[t == 1].var() + x[t == 0].var()) / 2)

t = df["app"].to_numpy()
# IPW estabilizado para el efecto medio (ATE)
p_t = t.mean()
w_ipw = np.where(t == 1, p_t / df["ps"], (1 - p_t) / (1 - df["ps"]))
equilibrio = pd.DataFrame({
    "SMD antes": [smd(df[c].to_numpy(float), t) for c in COVARIABLES],
    "SMD tras IPW": [smd(df[c].to_numpy(float), t, w_ipw) for c in COVARIABLES]}, index=COVARIABLES)
equilibrio

In [ ]:
# --- Emparejamiento 1-NN con reemplazo sobre el logit del ps: efecto en los tratados (ATT) -------
logit_ps = np.log(df["ps"] / (1 - df["ps"])).to_numpy().reshape(-1, 1)
tratados = np.where(t == 1)[0]
controles = np.where(t == 0)[0]
vecinos = NearestNeighbors(n_neighbors=1).fit(logit_ps[controles])
dist, idx = vecinos.kneighbors(logit_ps[tratados])
par = controles[idx.ravel()]
att = (df["gasto"].to_numpy()[tratados] - df["gasto"].to_numpy()[par]).mean()
informe("Emparejamiento por ps (ATT)", att, VERDAD)
print("Controles distintos usados:", len(set(par)), "de", len(controles))

# --- IPW: reponderar para que ambos grupos se parezcan a toda la población (ATE) ------------------
def ipw_ate(y, t, ps):
    return np.mean(t * y / ps) - np.mean((1 - t) * y / (1 - ps))

def ipw_hajek(y, t, ps):
    w1, w0 = t / ps, (1 - t) / (1 - ps)
    return np.sum(w1 * y) / np.sum(w1) - np.sum(w0 * y) / np.sum(w0)

y = df["gasto"].to_numpy()
informe("IPW (Horvitz-Thompson)", ipw_ate(y, t, df["ps"].to_numpy()), VERDAD)
informe("IPW normalizado (Hájek)", ipw_hajek(y, t, df["ps"].to_numpy()), VERDAD)

# --- Doblemente robusto (AIPW): modelo del resultado + ponderación ---------------------------------
def aipw(df, covs):
    X = sm.add_constant(df[covs])
    m1 = sm.OLS(df.loc[df.app == 1, "gasto"], X[df.app == 1]).fit()
    m0 = sm.OLS(df.loc[df.app == 0, "gasto"], X[df.app == 0]).fit()
    mu1, mu0 = m1.predict(X), m0.predict(X)
    ps = df["ps"]
    tt, yy = df["app"], df["gasto"]
    psi = mu1 - mu0 + tt * (yy - mu1) / ps - (1 - tt) * (yy - mu0) / (1 - ps)
    return psi.mean(), psi.std() / np.sqrt(len(psi))
est, ee = aipw(df, COVARIABLES)
informe("Doblemente robusto (AIPW)", est, VERDAD, (est - 1.96 * ee, est + 1.96 * ee))

# --- Incertidumbre: bootstrap que reestima TODO el procedimiento (también el ps) -------------------
rng_b = np.random.default_rng(0)
boots = []
for _ in range(100):
    s = df.sample(len(df), replace=True, random_state=int(rng_b.integers(1e9)))
    Xb = StandardScaler().fit_transform(s[COVARIABLES])
    psb = LogisticRegression(max_iter=500).fit(Xb, s["app"]).predict_proba(Xb)[:, 1]
    boots.append(ipw_hajek(s["gasto"].to_numpy(), s["app"].to_numpy(), psb))
print(f"IPW Hájek con bootstrap: IC95% [{np.percentile(boots, 2.5):.2f}, {np.percentile(boots, 97.5):.2f}]")

## DS1.6 · Diferencias en diferencias (DiD)

Veinte tiendas adoptan un nuevo escaparate en el mes 6; otras veinte, no. Las tratadas eran tiendas más grandes (nivel distinto), así que comparar niveles engaña. DiD compara **cambios**: el cambio de las tratadas menos el cambio del control. Supuesto clave: **tendencias paralelas** (sin tratamiento, ambos grupos habrían evolucionado igual).

In [ ]:
def generar_panel(n_tiendas=40, n_meses=12, mes_inicio=6, efecto=6.0, pendiente_dif=0.0, semilla=3):
    """Ventas mensuales (miles de €) de tiendas TiendaSur. La mitad adopta el nuevo escaparate en `mes_inicio`.
    Las tratadas son tiendas más grandes (nivel distinto) pero comparten tendencia, salvo `pendiente_dif`."""
    rng = np.random.default_rng(semilla)
    tratada = np.r_[np.ones(n_tiendas // 2), np.zeros(n_tiendas // 2)].astype(int)
    nivel = 50 + 12 * tratada + rng.normal(0, 4, n_tiendas)
    estacion = 3 * np.sin(2 * np.pi * np.arange(n_meses) / 12)
    filas = []
    for i in range(n_tiendas):
        for mes in range(n_meses):
            post = int(mes >= mes_inicio)
            ventas = (nivel[i] + 0.6 * mes + estacion[mes] + pendiente_dif * mes * tratada[i]
                      + efecto * post * tratada[i] + rng.normal(0, 1.5))
            filas.append((i, mes, tratada[i], post, ventas))
    return pd.DataFrame(filas, columns=["tienda", "mes", "tratada", "post", "ventas"])

panel = generar_panel()
medias = panel.groupby(["tratada", "post"])["ventas"].mean().unstack()
print(medias.round(2))
did_2x2 = (medias.loc[1, 1] - medias.loc[1, 0]) - (medias.loc[0, 1] - medias.loc[0, 0])
print(f"\nDiD a mano: (tratadas después - antes) - (control después - antes) = {did_2x2:.2f}")

m_did = smf.ols("ventas ~ tratada * post", panel).fit(cov_type="cluster", cov_kwds={"groups": panel["tienda"]})
b, se, lo, hi = coef(m_did, "tratada:post")
informe("DiD por regresión (errores por tienda)", b, 6.0, (lo, hi))
informe("Comparación ingenua post (tratadas - control)",
        panel.query("post==1 and tratada==1").ventas.mean() - panel.query("post==1 and tratada==0").ventas.mean(), 6.0)
informe("Antes-después solo en tratadas", medias.loc[1, 1] - medias.loc[1, 0], 6.0)

**Comprobar el supuesto.** No se puede probar, pero sí buscar evidencia en los periodos previos con un *estudio de eventos*: los coeficientes de los meses anteriores al tratamiento deberían ser ≈ 0.

In [ ]:
def estudio_eventos(panel, mes_inicio=6, referencia=5):
    """Coeficientes por mes relativo al inicio (ref. = último mes previo). Los previos deberían ser ≈ 0."""
    d = panel.copy()
    d["rel"] = d["mes"] - mes_inicio
    terminos = []
    for r in sorted(d["rel"].unique()):
        if r == referencia - mes_inicio:
            continue
        nombre = f"rel_{'m' if r < 0 else 'p'}{abs(r)}"
        d[nombre] = ((d["rel"] == r) & (d["tratada"] == 1)).astype(int)
        terminos.append(nombre)
    m = smf.ols("ventas ~ C(tienda) + C(mes) + " + " + ".join(terminos), d).fit(
        cov_type="cluster", cov_kwds={"groups": d["tienda"]})
    previos = [t for t in terminos if "_m" in t]
    wald = m.wald_test(", ".join(f"{p} = 0" for p in previos), scalar=True)
    tabla = pd.DataFrame({"coef": [m.params[t] for t in terminos], "lo": [m.conf_int().loc[t, 0] for t in terminos],
                          "hi": [m.conf_int().loc[t, 1] for t in terminos]},
                         index=[int(t[5:].replace("m", "-").replace("p", "")) for t in terminos]).sort_index()
    return tabla, float(wald.pvalue)

def dibujar_evento(tabla, titulo):
    fig, ax = plt.subplots()
    ax.errorbar(tabla.index, tabla["coef"], yerr=[tabla["coef"] - tabla["lo"], tabla["hi"] - tabla["coef"]], fmt="o", capsize=3)
    ax.axhline(0, color="grey", lw=.8); ax.axvline(-.5, color="red", ls="--", lw=.8)
    ax.set(xlabel="meses desde el inicio", ylabel="efecto estimado (k€)", title=titulo); plt.show()

ev, p_previos = estudio_eventos(panel)
dibujar_evento(ev, "Tendencias paralelas OK: los coeficientes previos rondan 0")
print(f"Test conjunto de tendencias previas: p = {p_previos:.3f}  (p alto: no se rechaza tendencia paralela)")

# Ahora violamos el supuesto: las tratadas ya crecían más deprisa (0,4 k€ por mes)
panel_mal = generar_panel(pendiente_dif=0.4)
m_mal = smf.ols("ventas ~ tratada * post", panel_mal).fit(cov_type="cluster", cov_kwds={"groups": panel_mal["tienda"]})
informe("DiD con tendencias NO paralelas", coef(m_mal, "tratada:post")[0], 6.0)
ev2, p2 = estudio_eventos(panel_mal)
dibujar_evento(ev2, "Tendencias NO paralelas: los previos ya suben")
print(f"Test de tendencias previas: p = {p2:.4f}  ->  alarma")

## DS1.7 · Variables instrumentales (2SLS)

Si hay un confusor **no observado** (la motivación), ajustar no sirve. Un **instrumento** Z mueve T pero solo afecta a Y *a través de* T. Aquí: una invitación por email enviada al azar para instalar la app.

Condiciones: **relevancia** (Z mueve T: se comprueba con el F de la primera etapa), **exogeneidad** (Z no se relaciona con los confusores: viene del azar o de argumentos) y **exclusión** (Z no afecta a Y salvo por T: se argumenta, no se prueba). El resultado es el efecto para los *cumplidores* (LATE).

In [ ]:
def generar_iv(n=20_000, fuerza=1.2, semilla=11, efecto=8.0):
    """U = motivación (NO observada) -> usa app y gasta más. Z = invitación aleatoria por email (instrumento)."""
    rng = np.random.default_rng(semilla)
    u = rng.normal(size=n)
    z = rng.binomial(1, 0.5, n)
    d = (-0.3 + fuerza * z + 1.0 * u + rng.logistic(size=n) > 0).astype(int)      # usa la app
    y = 50 + efecto * d + 6.0 * u + rng.normal(0, 10, n)
    return pd.DataFrame({"z": z, "d": d, "y": y, "u": u})

def dos_etapas(y, d, z, X=None):
    """2SLS 'a mano'. Devuelve (efecto, error estándar, F de la primera etapa)."""
    n = len(y)
    X = np.ones((n, 1)) if X is None else np.column_stack([np.ones(n), X])
    Z = np.column_stack([X, z])
    etapa1 = sm.OLS(d, Z).fit()
    d_hat = etapa1.fittedvalues
    D_hat = np.column_stack([X, d_hat])
    etapa2 = sm.OLS(y, D_hat).fit()
    # El error estándar correcto usa los residuos con la d REAL, no con d_hat
    D_real = np.column_stack([X, d])
    resid = y - D_real @ etapa2.params
    sigma2 = resid @ resid / (n - D_real.shape[1])
    cov = sigma2 * np.linalg.inv(D_hat.T @ D_hat)
    F = etapa1.tvalues[-1] ** 2
    return etapa2.params[-1], np.sqrt(cov[-1, -1]), F

iv = generar_iv()
VERDAD_IV = 8.0
b_ols = smf.ols("y ~ d", iv).fit()
informe("OLS ingenuo (U omitida)", b_ols.params["d"], VERDAD_IV)
wald = (iv.groupby("z")["y"].mean().diff().iloc[-1]) / (iv.groupby("z")["d"].mean().diff().iloc[-1])
informe("Estimador de Wald (efecto/cumplimiento)", wald, VERDAD_IV)
b, se, F = dos_etapas(iv["y"].to_numpy(), iv["d"].to_numpy(), iv["z"].to_numpy())
informe("2SLS", b, VERDAD_IV, (b - 1.96 * se, b + 1.96 * se))
print(f"F de la primera etapa = {F:,.0f}  (regla práctica: > 10)")
print("Comprobación: la invitación es independiente de la motivación -> corr(z, u) =", round(iv[['z', 'u']].corr().iloc[0, 1], 3))

# Instrumento DÉBIL: muchas repeticiones con n pequeño
def repetir(fuerza, reps=300, n=1500):
    est, Fs = [], []
    for r in range(reps):
        s = generar_iv(n=n, fuerza=fuerza, semilla=1000 + r)
        b, _, F = dos_etapas(s["y"].to_numpy(), s["d"].to_numpy(), s["z"].to_numpy())
        est.append(b); Fs.append(F)
    return np.array(est), np.array(Fs)

res_iv = {}
for nombre, f in [("fuerte (1,2)", 1.2), ("débil (0,08)", 0.08)]:
    est, Fs = repetir(f)
    res_iv[nombre] = {"F medio": Fs.mean(), "media estimaciones": est.mean(), "desv. típica": est.std(),
                      "p5": np.percentile(est, 5), "p95": np.percentile(est, 95)}
pd.DataFrame(res_iv).T

**Lee la tabla:** con instrumento fuerte la media está cerca de 8 y la dispersión es pequeña; con instrumento débil la dispersión se dispara y la media se contamina. De ahí la regla práctica F > 10.

## DS1.8 · Regresión discontinua (breve)

Cuando una regla asigna el tratamiento según un umbral («cupón de fidelidad si el gasto anual ≥ 100 €»), los clientes justo a ambos lados del corte son casi idénticos y la asignación es *como si* aleatoria. Se estima el salto con una regresión local. Supuesto: nadie manipula su posición respecto al corte.

In [ ]:
rng = np.random.default_rng(5)
n = 6000
gasto_anual = rng.normal(100, 30, n).clip(20, 200)
corte = 100.0
cupon = (gasto_anual >= corte).astype(int)         # regla del programa: cupón de fidelidad si gasto anual ≥ 100 €
EFECTO_RDD = 4.0
gasto_sig = 40 + 0.5 * gasto_anual + EFECTO_RDD * cupon + rng.normal(0, 8, n)
rd = pd.DataFrame({"x": gasto_anual - corte, "cupon": cupon, "y": gasto_sig})

informe("Diferencia de medias (ingenua)", rd.query("cupon==1").y.mean() - rd.query("cupon==0").y.mean(), EFECTO_RDD)

def rdd_local(rd, h):
    """Regresión lineal local con núcleo triangular, pendientes distintas a cada lado del corte."""
    d = rd[rd["x"].abs() <= h].copy()
    d["w"] = 1 - d["x"].abs() / h
    m = smf.wls("y ~ cupon + x + cupon:x", d, weights=d["w"]).fit(cov_type="HC1")
    return coef(m, "cupon"), len(d)

filas = []
for h in [5, 10, 20, 40]:
    (b, se, lo, hi), nn = rdd_local(rd, h)
    filas.append({"ancho de banda": h, "n": nn, "efecto": b, "IC95 inf": lo, "IC95 sup": hi})
print(pd.DataFrame(filas).to_string(index=False))

fig, ax = plt.subplots()
bins = pd.cut(rd["x"], np.arange(-80, 81, 5))
medias = rd.groupby(bins, observed=True).agg(x=("x", "mean"), y=("y", "mean"))
ax.scatter(medias["x"], medias["y"], s=18); ax.axvline(0, color="red", ls="--", lw=.8)
ax.set(xlabel="gasto anual - 100 €", ylabel="gasto siguiente (media por tramo)", title="Discontinuidad en el corte"); plt.show()
print("Clientes justo por debajo / por encima del corte (±5 €):",
      int(((rd.x < 0) & (rd.x >= -5)).sum()), "/", int(((rd.x >= 0) & (rd.x < 5)).sum()), "(sin manipulación deberían ser parecidos)")

## DS1.9 · *Uplift modeling*: ¿a quién enviamos el cupón?

Un modelo de respuesta predice *quién comprará*. Pero el cupón solo vale la pena donde **cambia** la decisión. Cuatro tipos de cliente: **persuadibles** (compran solo con cupón), **seguros** (comprarían igual: el cupón es dinero regalado), **causas perdidas** (no compran) y **«perros dormidos»** (el cupón les molesta). El *uplift* es el efecto causal individual: P(compra | cupón) − P(compra | sin cupón).

Como el experimento está aleatorizado, podemos entrenar y, sobre todo, **evaluar** sin sesgo.

In [ ]:
def generar_experimento(n=60_000, semilla=21):
    """Experimento ALEATORIO: la mitad recibe el cupón (5 € de descuento). Resultado: ¿compra en 30 días?
    El efecto del cupón depende del cliente: sensibles al precio (+), fieles que ya compraban (≈0 o −)."""
    rng = np.random.default_rng(semilla)
    d = pd.DataFrame({
        "recencia": rng.integers(1, 120, n),          # días desde la última compra
        "frecuencia": rng.poisson(4, n),
        "gasto_prev": rng.lognormal(3.5, 0.6, n),
        "sensible_precio": rng.binomial(1, 0.35, n),
        "antiguedad": rng.integers(0, 72, n),
    })
    base = -1.0 - 0.015 * d["recencia"] + 0.25 * d["frecuencia"] + 0.003 * d["gasto_prev"]
    b = 1.1 * d["sensible_precio"] * (d["recencia"] > 30) - 0.5 * (d["frecuencia"] >= 7) + 0.1
    p0 = 1 / (1 + np.exp(-base))
    p1 = 1 / (1 + np.exp(-(base + b)))
    d["tratado"] = rng.binomial(1, 0.5, n)
    d["compra"] = rng.binomial(1, np.where(d["tratado"] == 1, p1, p0))
    d["uplift_real"] = p1 - p0
    return d

exp = generar_experimento()
CARACT = ["recencia", "frecuencia", "gasto_prev", "sensible_precio", "antiguedad"]
ate = exp.query("tratado==1").compra.mean() - exp.query("tratado==0").compra.mean()
print(f"Conversión con cupón {exp.query('tratado==1').compra.mean():.1%} | sin cupón {exp.query('tratado==0').compra.mean():.1%} | ATE = {ate:+.2%}")
print("Uplift real: media", f"{exp.uplift_real.mean():+.3f}", "| mínimo", f"{exp.uplift_real.min():+.3f}", "| máximo", f"{exp.uplift_real.max():+.3f}",
      "| clientes con efecto NEGATIVO:", f"{(exp.uplift_real < 0).mean():.1%}")
entreno, prueba = train_test_split(exp, test_size=0.4, random_state=0)
print(len(entreno), "para entrenar,", len(prueba), "para evaluar (aleatorizado, así que la evaluación es honesta)")

In [ ]:
def modelo_gb():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.08, max_iter=150, random_state=0)

# S-learner: UN modelo con el tratamiento como variable más
s = modelo_gb().fit(entreno[CARACT + ["tratado"]], entreno["compra"])
def pred_s(d):
    return (s.predict_proba(d[CARACT].assign(tratado=1))[:, 1] - s.predict_proba(d[CARACT].assign(tratado=0))[:, 1])

# T-learner: DOS modelos (tratados y controles); uplift = diferencia de predicciones
mt = modelo_gb().fit(entreno.query("tratado==1")[CARACT], entreno.query("tratado==1")["compra"])
mc = modelo_gb().fit(entreno.query("tratado==0")[CARACT], entreno.query("tratado==0")["compra"])
def pred_t(d):
    return mt.predict_proba(d[CARACT])[:, 1] - mc.predict_proba(d[CARACT])[:, 1]

# Modelo de respuesta "clásico": ¿quién comprará si le envío el cupón? (NO es uplift)
def pred_resp(d):
    return mt.predict_proba(d[CARACT])[:, 1]

prueba = prueba.copy()
prueba["u_s"], prueba["u_t"], prueba["u_resp"] = pred_s(prueba), pred_t(prueba), pred_resp(prueba)
print("Correlación con el uplift REAL (solo posible porque es una simulación):")
print(prueba[["u_s", "u_t", "u_resp"]].corrwith(prueba["uplift_real"]).round(3).to_string())

**Curva Qini.** Ordena a los clientes de mayor a menor uplift predicho y acumula las compras *incrementales* al ir tratando a más. Un buen modelo sube rápido y luego cae (los últimos son «perros dormidos»). El área sobre la línea del azar es el AUUC.

In [ ]:
def curva_qini(y, t, puntuacion, n_puntos=50):
    """Uplift acumulado (en nº de compras incrementales) al tratar al top-x % según `puntuacion`."""
    y, t, puntuacion = map(np.asarray, (y, t, puntuacion))
    orden = np.argsort(-puntuacion)
    y, t = y[orden], t[orden]
    n = len(y)
    fr, q = [0.0], [0.0]
    for k in np.linspace(n / n_puntos, n, n_puntos).astype(int):
        yt, yc = y[:k][t[:k] == 1].sum(), y[:k][t[:k] == 0].sum()
        nt, nc = (t[:k] == 1).sum(), (t[:k] == 0).sum()
        fr.append(k / n)
        q.append(yt - yc * nt / max(nc, 1))
    return np.array(fr), np.array(q)

def auuc(fr, q):
    """Área entre la curva Qini del modelo y la línea del azar (tratar al azar)."""
    azar = q[-1] * fr
    return np.trapezoid(q - azar, fr)

fig, ax = plt.subplots(figsize=(7.5, 4))
resultados = {}
for nombre, col in [("S-learner", "u_s"), ("T-learner", "u_t"), ("modelo de respuesta", "u_resp"),
                    ("oráculo (uplift real)", "uplift_real"), ("azar", None)]:
    punt = prueba[col] if col else np.random.default_rng(0).random(len(prueba))
    fr, q = curva_qini(prueba["compra"], prueba["tratado"], punt)
    resultados[nombre] = auuc(fr, q)
    ax.plot(fr, q, label=nombre, ls="--" if col is None else "-")
ax.set(xlabel="fracción de clientes tratados (ordenados por uplift)", ylabel="compras incrementales", title="Curvas Qini (conjunto de prueba)")
ax.legend(); plt.show()
print(pd.Series(resultados, name="AUUC (mayor = mejor)").round(1).to_string())

**De la curva a la decisión.** El coste del cupón se paga en cada compra con cupón, incluidas las de clientes «seguros». Ese es justo el dinero que el uplift ayuda a no malgastar.

In [ ]:
MARGEN = 14.0      # € de margen por compra (sin cupón)
CUPON = 5.0        # € de descuento que se paga en cada compra con cupón

def beneficio_incremental(y, t, puntuacion, fraccion, margen=MARGEN, cupon=CUPON):
    """Beneficio incremental (€ por cada 1 000 clientes del total) de enviar el cupón solo al top `fraccion`,
    estimado con el experimento: (margen - cupón) · compras de los tratados − margen · compras que esos mismos
    clientes habrían hecho sin cupón (estimadas con el grupo de control)."""
    y, t, puntuacion = map(np.asarray, (y, t, puntuacion))
    top = np.argsort(-puntuacion)[: int(fraccion * len(y))]
    yy, tt = y[top], t[top]
    nt, nc = (tt == 1).sum(), (tt == 0).sum()
    compras_t = yy[tt == 1].sum()
    compras_c_escaladas = yy[tt == 0].sum() * nt / nc
    por_cliente_tratado = ((margen - cupon) * compras_t - margen * compras_c_escaladas) / nt
    return por_cliente_tratado * fraccion * 1000          # los no seleccionados no cuestan ni aportan nada

fracciones = np.linspace(0.05, 1.0, 20)
curvas = {nombre: [beneficio_incremental(prueba["compra"], prueba["tratado"], prueba[col], f) for f in fracciones]
          for nombre, col in [("T-learner", "u_t"), ("modelo de respuesta", "u_resp")]}
fig, ax = plt.subplots()
for nombre, vals in curvas.items():
    ax.plot(fracciones, vals, label=nombre)
ax.axhline(0, color="grey", lw=.8); ax.set(xlabel="fracción de clientes a los que se envía el cupón", ylabel="beneficio incremental (€ por 1 000 clientes)",
                                           title="¿A cuántos enviamos el cupón?"); ax.legend(); plt.show()
mejor = int(np.argmax(curvas["T-learner"]))
print(f"Óptimo con T-learner: enviar al {fracciones[mejor]:.0%} de los clientes -> {curvas['T-learner'][mejor]:,.0f} € por 1 000 clientes")
print(f"Enviar a TODOS: {curvas['T-learner'][-1]:,.0f} € por 1 000 clientes")

## Resumen: qué método recuperó el efecto

Repasa las líneas «verdad» frente a «estimación» de cada sección. Regla de oro: **el método lo decide el diseño de los datos** (¿hay confusores observados? ¿hay un antes/después con control? ¿hay un instrumento? ¿hay un umbral? ¿hay un experimento?), no la moda del momento.

## Ejercicios

Resuelve cada ejercicio en la celda que sigue. Las funciones y los datos de las secciones anteriores ya están disponibles.

1. **DAG y conjunto de ajuste.** TiendaSur quiere medir el efecto de la *promoción en tienda* sobre las *ventas*. Dibuja (como lista de aristas) este DAG: `clima → promocion`, `clima → ventas`, `promocion → ventas`, `promocion → trafico`, `trafico → ventas`, `promocion → quejas`, `ventas → quejas`. Usa `criterio_puerta_trasera` con cuatro conjuntos candidatos (`{}`, `{clima}`, `{clima, trafico}`, `{clima, quejas}`) y explica en un comentario cuál es el mínimo válido y por qué los otros fallan.
2. **Solapamiento pobre.** Repite 100 veces `generar_clientes(n=4000, fuerza_seleccion=3.0, semilla=s)`. En cada una estima el ATE con IPW normalizado (Hájek), con los pesos sin recortar y recortando el *propensity score* a [0,05; 0,95]. Compara sesgo y desviación típica con el efecto real (8). ¿Qué compromiso introduce el recorte?
3. **Tendencias no paralelas.** Con `generar_panel(pendiente_dif=0.4)`, estima el efecto con DiD simple y con una **tendencia lineal propia por grupo** (`ventas ~ C(tienda) + C(mes) + tratada:mes + tratada:post`). ¿Cuál se acerca más al efecto real (6)? ¿Qué supuesto hay ahora que justificar?
4. **Instrumento débil.** Con `generar_iv(n=1500, fuerza=0.08)` calcula el F de la primera etapa y la estimación 2SLS para 5 semillas distintas. ¿Por qué no te fiarías de ninguna aunque el instrumento sea «válido» por construcción?
5. **Decisión de negocio con el uplift.** Si el cupón costara 6 € en vez de 5 € (y el margen fuera 14 €), ¿a qué fracción de clientes enviarías el cupón según el T-learner? Calcula el beneficio incremental por 1 000 clientes en el óptimo y compáralo con enviarlo a todos.


In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
